# ST456 One-Click Colab Notebook (ZIP Upload)

Use this notebook to run the current mainline project setup:

- Run E1-E5 as the main experiments
- Keep retrieval in the appendix
- Skip GitHub
- Upload one ZIP file and click `Runtime -> Run all`


In [ ]:
# Edit these settings before you run the notebook
ZIP_EXPECTED_HINT = 'ST456group.zip'
PROJECT_SUBDIR = 'text'

USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_WORKSPACE = '/content/drive/MyDrive/st456_runs'

# Run the full mainline by default: full training schedule and full test-set evaluation
QUICK_VALIDATION = False

# Run E1-E5 in one pass
RUN_FULL_MAINLINE = True

# Run the retrieval appendix by default
RUN_APPENDIX_RETRIEVAL = True

# Package and download the result bundle
DOWNLOAD_RESULTS_ZIP = True

print('Settings loaded. This notebook will ask you for a ZIP file.')

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

# Keep model downloads from flooding the cell output.
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    """Run a command and keep one live status line on screen."""
    import html
    from collections import deque
    from IPython.display import HTML, display

    print(f'\n>>> {cmd}', flush=True)
    status_handle = display(
        HTML(f"<pre>>>> {html.escape(cmd)}\n{html.escape('Starting...')}</pre>"),
        display_id=True,
    )
    tail = deque(maxlen=80)

    def update_status(text):
        safe_cmd = html.escape(cmd)
        safe_text = html.escape(text or 'Running...')
        status_handle.update(HTML(f"<pre>>>> {safe_cmd}\n{safe_text}</pre>"))

    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )

    current = ''
    while True:
        char = process.stdout.read(1)
        if char == '' and process.poll() is not None:
            break
        if not char:
            continue
        if char == '\r':
            line = current.strip()
            if line:
                update_status(line[-400:])
            current = ''
            continue
        if char == '\n':
            line = current.strip()
            if line:
                tail.append(line)
                update_status(line[-400:])
            current = ''
            continue
        current += char

    final_line = current.strip()
    if final_line:
        tail.append(final_line)
        update_status(final_line[-400:])

    process.wait()
    if process.returncode != 0:
        recent_output = '\n'.join(tail) or 'No output captured.'
        update_status(f'Failed (exit code {process.returncode})')
        raise RuntimeError(
            f'Command failed (exit code {process.returncode}): {cmd}\n'
            f'Recent output:\n{recent_output}'
        )

    update_status('Done')

def clear_gpu_memory(stage=''):
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        print(f'[GPU cleanup] {stage or "Current stage"}: torch is not installed, so this step was skipped.')
        return

    if not torch.cuda.is_available():
        print(f'[GPU cleanup] {stage or "Current stage"}: CUDA is not available, so this step was skipped.')
        return

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, 'ipc_collect'):
        torch.cuda.ipc_collect()
    reserved_mb = torch.cuda.memory_reserved() / (1024 ** 2)
    allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    print(
        f'[GPU cleanup] {stage or "Current stage"}: '        f'allocated={allocated_mb:.1f} MB, reserved={reserved_mb:.1f} MB'
    )

workspace_root = Path('/content')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    workspace_root = Path(GOOGLE_DRIVE_WORKSPACE)

workspace_root.mkdir(parents=True, exist_ok=True)
os.chdir(workspace_root)
print('Current working directory:', Path.cwd())

from google.colab import files
print(f'Upload your project ZIP file, for example: {ZIP_EXPECTED_HINT}')
uploaded = files.upload()
zip_candidates = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('No ZIP file was uploaded.')

zip_name = zip_candidates[0]
print('Uploaded ZIP:', zip_name)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(workspace_root)

project_matches = sorted(workspace_root.rglob(PROJECT_SUBDIR))
project_matches = [path for path in project_matches if path.is_dir()]
if not project_matches:
    raise FileNotFoundError(f'Could not find the {PROJECT_SUBDIR} directory after extraction.')

project_root = project_matches[0]
os.chdir(project_root)
src_root = project_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
print('Project directory:', Path.cwd())
print('Added to Python path:', src_root)
run(f'{sys.executable} -m pip install -r requirements.txt')
print('Environment setup is complete.')

## Build the data and inspect token budgets

This step will:

- Download the Sherlock Holmes corpus
- Build the processed dataset
- Save token-budget summaries for E1, E3, and E5


In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]

for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token statistics files:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)

## Train the mainline experiments

This notebook always runs E1. It will keep going through E2-E5 when `RUN_FULL_MAINLINE = True`.


In [ ]:
from novel_continuation.training import load_training_config

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
    ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
    ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ('E5W', 'configs/e5_distilgpt2_structured_aux_ranking_wide.yaml', 'artifacts/e5_aux_ranking_wide'),
]

for experiment_id, config_path, _output_dir in main_experiments:
    print(f'----- {experiment_id}: training starts -----')
    run(f'{sys.executable} scripts/train_experiment.py --config {config_path} --seed 42')
    clear_gpu_memory(f'{experiment_id} training finished')

print('Mainline training is complete.')
for experiment_id, config_path, _output_dir in main_experiments:
    resolved = load_training_config(Path(config_path))
    print(f'- {experiment_id}: output_dir={resolved["output_dir"]}')

## Run 3-seed generation and automatic evaluation

This step writes the main evaluation outputs for each trained experiment:

- `generated_samples_*_seed*.jsonl`
- `metrics_*_seed*.csv`
- `metrics_*_summary.csv`
- `human_eval_*.csv` if you decide to export the optional appendix file


In [ ]:
import json
eval_output_dir = Path('artifacts/eval')
eval_output_dir.mkdir(parents=True, exist_ok=True)

for experiment_id, _config_path, model_dir in main_experiments:
    print(f'\n----- {experiment_id}: 3-seed automatic evaluation -----')
    run(
        f'{sys.executable} scripts/run_eval_3seed.py ' \
        f'--experiment-id {experiment_id.lower()} ' \
        f'--model-dir {model_dir} ' \
        f'--output-dir {eval_output_dir}'
    )
    clear_gpu_memory(f'{experiment_id} evaluation finished')

print('\nThe 3-seed evaluation files are ready.')
for path in sorted(eval_output_dir.glob('*')):
    print('-', path)

print('\nCompare validation_main_loss for E5 and E5W:')
for experiment_id, _config_path, model_dir in main_experiments:
    if experiment_id not in {'E5', 'E5W'}:
        continue
    training_config = Path(model_dir) / 'training_config.json'
    if not training_config.exists():
        print(f'- {experiment_id}: could not find {training_config}')
        continue
    payload = json.loads(training_config.read_text(encoding='utf-8'))
    validation = payload.get('metadata', {}).get('validation', {})
    print(f'- {experiment_id}: validation_main_loss={validation.get("validation_main_loss")}, validation_perplexity={validation.get("validation_perplexity")}')

## Appendix: retrieval experiment

This block only runs when `RUN_APPENDIX_RETRIEVAL = True`.


In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    # Keep human-eval export as an appendix step instead of part of the default run.
    print('The retrieval appendix run is complete.')
else:
    print('The notebook skipped the retrieval appendix run.')

## Package and download the results

When `DOWNLOAD_RESULTS_ZIP = True`, this step will:

1. Remove intermediate checkpoints to save space
2. Package evaluation outputs and training configs without the model weights


In [ ]:
import glob

if DOWNLOAD_RESULTS_ZIP:
    # Remove intermediate checkpoints to save Colab disk space.
    for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
        if ckpt_dir.is_dir():
            shutil.rmtree(ckpt_dir)
            print(f'Removed: {ckpt_dir}')

    # Collect the files you will want after the run.
    download_dir = Path('artifacts/download_package')
    download_dir.mkdir(parents=True, exist_ok=True)

    # Copy evaluation outputs such as metrics, generations, human-eval CSVs, and token stats.
    eval_src = Path('artifacts/eval')
    if eval_src.exists():
        shutil.copytree(eval_src, download_dir / 'eval', dirs_exist_ok=True)

    # Copy each training_config.json so you keep the run metadata for the report.
    for config_file in Path('artifacts').glob('*/training_config.json'):
        dest = download_dir / config_file.parent.name / config_file.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_file, dest)

    # Build the ZIP file and download it.
    archive_path = shutil.make_archive(
        str(Path('artifacts') / 'colab_results'), 'zip',
        root_dir=str(download_dir)
    )
    size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f'ZIP file created: {archive_path} ({size_mb:.1f} MB)')

    # Remove the temporary staging directory.
    shutil.rmtree(download_dir)

    from google.colab import files
    files.download(archive_path)
else:
    print('The notebook skipped the result download step.')